# U-Net Glomeruli Segmentation — Binary Segmentation Training Pipeline

This notebook implements a complete PyTorch U-Net training system for binary semantic segmentation of glomeruli from renal biopsy whole-slide images.

## Two-Model Strategy

This pipeline uses a **two-stage approach** for glomeruli classification:

1. **Model 1 (this notebook)**: Binary segmentation to detect glomerulus locations
   - Input: Z-score normalized RGB tiles from WSI
   - Output: Pixel-level binary masks (background vs glomerulus)
   - Classes: 0 (background), 1 (glomerulus — fused from original classes 1-4)
   - Purpose: Localizes all glomeruli regardless of subtype

2. **Model 2 (future)**: Classifier on detected patches to assign classes 1-4
   - Input: Image patches extracted from detected glomerulus regions (Model 1)
   - Output: Per-glomerulus classification (classes 1-4: FGS, MPGN, FSGS, other)
   - Purpose: Fine-grained subtype classification for clinical diagnosis

## Notebook Contents

1. **Data Loading** — GlomeruliDataset with stratified split and online augmentation (train only)
2. **Loss Functions** — Dice Loss + Cross-Entropy + Class Weighting
3. **U-Net Architecture** — Encoder-Decoder with Skip Connections
4. **Binary Segmentation Metrics** — Accuracy, Precision, Recall, F1, ROC-AUC
5. **Training Pipeline** — Full training loop with validation and checkpointing

## Pipeline Overview

WSI TIFF + GeoJSON annotations → Tiled images + masks (classes 0-4) →
Reinhard color normalization → Z-score standardization →
Train/Val/Test split (stratified by biopsy, 70/15/15) →
Online augmentation (train only) → U-Net training → Model evaluation (Binary Metrics)

See WORKFLOW.md for end-to-end pipeline documentation.

In [ ]:
# Standard library
import os
import json
import argparse
import logging
from pathlib import Path
from datetime import datetime
from typing import Tuple, Optional, List

# Scientific computing
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR
from torch.utils.tensorboard import SummaryWriter
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Data loading
import cv2
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
import warnings

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Part 1: Data Loading Architecture

Loads paired (image, mask) data from the preprocessing pipeline.

### Key Concepts

- **Stratified split by biopsia**: Prevents data leakage by ensuring tiles from same slide don't appear in both train and test
- **Z-score decoding**: Converts uint8 PNG (0-255) back to float32 (~-4 to +4)
- **Optional augmentation**: albumentations on training set only

In [ ]:
class GlomeruliDataset(Dataset):
    """
    Dataset for glomeruli segmentation with binary mask conversion.

    Args:
        images_dir: Path to Salidas/Estandarizados/ (Z-score encoded tiles)
        masks_dir: Path to masks directory. If None, auto-detects as same root as images_dir (default: None)
        split: 'train', 'val', or 'test' (mutually exclusive)
        train_size: Fraction of biopsies for training (default 0.70)
        val_size: Fraction of biopsies for validation (default 0.15)
        seed: Random seed for reproducible splits
        transforms: Optional albumentations Compose for augmentation (train only)
    """

    def __init__(
        self,
        images_dir: str,
        masks_dir: str = None,
        split: str = 'train',
        train_size: float = 0.70,
        val_size: float = 0.15,
        seed: int = 42,
        transforms=None,
    ):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir) if masks_dir is not None else Path(images_dir)
        self.split = split
        self.transforms = transforms

        assert split in {'train', 'val', 'test'}, f"Invalid split: {split}"
        assert self.images_dir.exists(), f"Images dir not found: {self.images_dir}"
        assert self.masks_dir.exists(), f"Masks dir not found: {self.masks_dir}"

        test_size = 1.0 - train_size - val_size
        assert test_size >= 0, "train_size + val_size must be <= 1.0"

        # Scan all images and group by biopsia (top-level subdirectory)
        self.all_images = sorted(self.images_dir.rglob('*.png'))

        if not self.all_images:
            raise ValueError(f"No PNG images found in {self.images_dir}")

        # Group image paths by biopsia (first component of relative path)
        biopsias_dict = {}
        for img_path in self.all_images:
            relative = img_path.relative_to(self.images_dir)
            biopsia = relative.parts[0]
            if biopsia not in biopsias_dict:
                biopsias_dict[biopsia] = []
            biopsias_dict[biopsia].append(img_path)

        biopsias_list = list(biopsias_dict.keys())

        # Split biopsias (not tiles) to avoid data leakage
        rng = np.random.RandomState(seed)

        # First split: train + val vs test
        train_val_biopsias, test_biopsias = train_test_split(
            biopsias_list,
            test_size=test_size if test_size > 0 else None,
            random_state=seed,
        )

        # Second split: train vs val
        if val_size > 0:
            val_fraction = val_size / (train_size + val_size)
            train_biopsias, val_biopsias = train_test_split(
                train_val_biopsias,
                test_size=val_fraction,
                random_state=seed + 1,
            )
        else:
            train_biopsias = train_val_biopsias
            val_biopsias = []

        # Collect tiles for this split
        if split == 'train':
            selected_biopsias = train_biopsias
        elif split == 'val':
            selected_biopsias = val_biopsias
        else:  # test
            selected_biopsias = test_biopsias

        self.image_paths = []
        for biopsia in selected_biopsias:
            self.image_paths.extend(biopsias_dict[biopsia])

        self.image_paths = sorted(self.image_paths)

        # Verify all images have corresponding masks
        self.mask_paths = []
        missing = []
        for img_path in self.image_paths:
            mask_path = self._get_mask_path(img_path)
            if mask_path.exists():
                self.mask_paths.append(mask_path)
            else:
                missing.append((img_path, mask_path))

        if missing:
            warnings.warn(
                f"Found {len(missing)} images without corresponding masks. "
                f"These will be skipped. First few: {missing[:3]}"
            )
            # Remove unpaired images
            paired = [
                (img, msk)
                for img, msk in zip(self.image_paths, self.mask_paths)
                if (self.masks_dir / msk).exists()
            ]
            if paired:
                self.image_paths, self.mask_paths = zip(*paired)
                self.image_paths = list(self.image_paths)
                self.mask_paths = list(self.mask_paths)
            else:
                raise ValueError("No valid image-mask pairs found after checking.")

    def _get_mask_path(self, image_path: Path) -> Path:
        """
        Convert image path to mask path using naming convention.

        Example:
          images: Salidas/Estandarizados/biopsia_001/images/slide_tile_0_1.png
          masks:  Salidas/dataset_aug/biopsia_001/masks/slide_tile_0_1_mask.png
        """
        # Get relative path from images_dir
        rel = image_path.relative_to(self.images_dir)

        # Build corresponding path in masks_dir
        # Replace 'images' with 'masks' and append '_mask' before .png
        parts = list(rel.parts)
        parts[1] = 'masks'  # images -> masks

        stem = parts[-1].replace('.png', '')
        parts[-1] = f"{stem}_mask.png"

        return self.masks_dir / Path(*parts)

    def _decode_zscore_png(self, img_uint8: np.ndarray) -> np.ndarray:
        """
        Decode Z-score from uint8 back to float32.

        Inverse of guardar_imagen_zscore_png in estandarizacion.py:
          uint8 = (float32 + 4) / 8 * 255

        So: float32 = (uint8 / 255 * 8) - 4
        """
        img_float = img_uint8.astype(np.float32) / 255.0
        img_float = (img_float * 8.0) - 4.0
        return img_float

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Returns (image, mask) pair with binary segmentation.

        image: torch.float32, shape [3, H, W], range ~[-4, +4] (Z-score)
        mask:  torch.long,    shape [H, W], values 0-1 (0=background, 1=glomerulus)
        """
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        # Load image (Z-score encoded as uint8)
        img_bgr = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise RuntimeError(f"Failed to load image: {img_path}")

        # Convert channels first (uint8), then decode Z-score once
        img_rgb_uint8 = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_rgb = self._decode_zscore_png(img_rgb_uint8)

        # Load mask (class IDs, uint8, values 0-4)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise RuntimeError(f"Failed to load mask: {mask_path}")

        # Convert to binary: fuse classes 1-4 into class 1
        # This maintains spatial structure of glomeruli while simplifying to binary task
        mask_binary = (mask > 0).astype(np.uint8)  # 0 stays 0, any non-zero becomes 1

        # Apply augmentations (if transforms provided and split is train)
        if self.transforms is not None and self.split == 'train':
            augmented = self.transforms(image=img_rgb, mask=mask_binary)
            img_rgb = augmented['image']
            mask_binary = augmented['mask']

        # Convert to tensors
        # Image: [H, W, 3] -> [3, H, W], float32
        img_tensor = torch.from_numpy(np.transpose(img_rgb, (2, 0, 1))).float()

        # Mask: [H, W], uint8 -> long (for CrossEntropyLoss)
        mask_tensor = torch.from_numpy(mask_binary.astype(np.int64)).long()

        return img_tensor, mask_tensor

In [ ]:
def create_dataloaders(
    images_dir: str,
    masks_dir: str = None,
    batch_size: int = 4,
    num_workers: int = 4,
    seed: int = 42,
    train_transforms=None,
    val_transforms=None,
):
    """
    Create train/val/test DataLoaders with proper stratification by biopsia.

    Returns:
        (train_loader, val_loader, test_loader)
    """
    train_ds = GlomeruliDataset(
        images_dir,
        masks_dir,
        split='train',
        seed=seed,
        transforms=train_transforms,
    )

    val_ds = GlomeruliDataset(
        images_dir,
        masks_dir,
        split='val',
        seed=seed,
        transforms=val_transforms,
    )

    test_ds = GlomeruliDataset(
        images_dir,
        masks_dir,
        split='test',
        seed=seed,
        transforms=None,  # Never augment test set
    )

    train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )

    val_loader = torch.utils.data.DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    test_loader = torch.utils.data.DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    return train_loader, val_loader, test_loader

In [ ]:
# Quick test of dataset loading
if Path('Salidas/Estandarizados').exists():
    ds_test = GlomeruliDataset('Salidas/Estandarizados', split='train')
    print(f"Dataset loaded: {len(ds_test)} tiles")
    img, mask = ds_test[0]
    print(f"Image shape: {img.shape}, dtype: {img.dtype}")
    print(f"Mask shape: {mask.shape}, dtype: {mask.dtype}")
    print(f"Mask unique classes (binary): {torch.unique(mask).tolist()}")
    print(f"Class distribution: 0 (background)={torch.sum(mask == 0).item()}, 1 (glomerulus)={torch.sum(mask == 1).item()}")
else:
    print("Dataset directory not found. Run preprocessing pipeline first.")
    print("Required: tiling_unet.py -> normalizacion.py -> estandarizacion.py")

## Part 2: Loss Functions for Class Imbalance

Semantic segmentation of medical images has extreme class imbalance: most pixels are background.

### Loss Strategy: BCE + Dice

- **Cross-Entropy Loss**: Strict per-pixel classification
- **Dice Loss**: Penalizes lack of geometric overlap (Dice coefficient = 2|X∩Y|/(|X|+|Y|))
- **Class Weighting**: Inverse frequency weights (rare glomerulus classes weighted higher)
- **Combined**: 50/50 weighted sum

In [ ]:
class DiceLoss(nn.Module):
    """
    Dice Loss for semantic segmentation.

    Dice = 2 * |X ∩ Y| / (|X| + |Y|)
    DiceLoss = 1 - Dice

    Computes per-class Dice and averages across classes (excluding background optionally).
    """

    def __init__(self, num_classes: int, smooth: float = 1e-6, ignore_background: bool = False):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
        self.ignore_background = ignore_background

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Args:
            logits: [B, C, H, W] model output (logits, no activation)
            targets: [B, H, W] ground truth (class IDs, dtype long)

        Returns:
            Scalar loss value
        """
        # Convert logits to probabilities
        probs = torch.nn.functional.softmax(logits, dim=1)  # [B, C, H, W]

        # One-hot encode targets
        targets_one_hot = torch.nn.functional.one_hot(targets, num_classes=self.num_classes)  # [B, H, W, C]
        targets_one_hot = targets_one_hot.permute(0, 3, 1, 2).float()  # [B, C, H, W]

        # Compute Dice per class
        dice_scores = []
        for c in range(self.num_classes):
            if self.ignore_background and c == 0:
                continue

            pred_c = probs[:, c, :, :]  # [B, H, W]
            target_c = targets_one_hot[:, c, :, :]  # [B, H, W]

            intersection = (pred_c * target_c).sum()
            union = pred_c.sum() + target_c.sum()

            dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
            dice_scores.append(dice)

        # Mean Dice across selected classes
        if dice_scores:
            mean_dice = torch.stack(dice_scores).mean()
        else:
            mean_dice = torch.tensor(0.0, device=logits.device)

        return 1.0 - mean_dice

In [ ]:
class CombinedLoss(nn.Module):
    """
    Combined CrossEntropy + Dice Loss for semantic segmentation.

    Handles class imbalance via weight vector on CE and Dice.
    """

    def __init__(
        self,
        num_classes: int = 5,
        weight_ce: float = 0.5,
        weight_dice: float = 0.5,
        class_weights: torch.Tensor = None,
        smooth: float = 1e-6,
        ignore_background: bool = False,
    ):
        """
        Args:
            num_classes: Number of output classes
            weight_ce: Weight for CrossEntropy term
            weight_dice: Weight for Dice term
            class_weights: [C] tensor of per-class weights (e.g., inverse frequency)
                           If None, uniform weights
            smooth: Smoothing constant for Dice
            ignore_background: If True, exclude class 0 from Dice loss
        """
        super().__init__()
        self.num_classes = num_classes
        self.weight_ce = weight_ce
        self.weight_dice = weight_dice
        self.smooth = smooth

        # Register class weights as buffer so they move with model
        if class_weights is not None:
            self.register_buffer('class_weights', class_weights)
        else:
            self.register_buffer('class_weights', torch.ones(num_classes))

        self.ce_loss = nn.CrossEntropyLoss(weight=self.class_weights)
        self.dice_loss = DiceLoss(
            num_classes=num_classes,
            smooth=smooth,
            ignore_background=ignore_background,
        )

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Args:
            logits: [B, C, H, W]
            targets: [B, H, W] with class IDs

        Returns:
            Scalar loss
        """
        ce = self.ce_loss(logits, targets)
        dice = self.dice_loss(logits, targets)
        loss = self.weight_ce * ce + self.weight_dice * dice
        return loss

In [ ]:
def compute_class_weights(masks_dir: str, num_classes: int = 2) -> torch.Tensor:
    """
    Compute per-class weights based on inverse frequency from all masks.

    For binary segmentation, this accounts for the extreme class imbalance
    where background (class 0) dominates glomerulus pixels (class 1).

    Args:
        masks_dir: Path to directory containing mask PNG files (recursive scan)
        num_classes: Number of classes (default 2 for binary segmentation)

    Returns:
        [num_classes] tensor of class weights (normalized to sum to num_classes)
    """
    masks_dir = Path(masks_dir)
    class_counts = np.zeros(num_classes)

    # Scan all masks
    for mask_path in masks_dir.rglob('*_mask.png'):
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            continue
        # Convert to binary: any non-zero is class 1
        mask_binary = (mask > 0).astype(np.uint8)
        for c in range(num_classes):
            class_counts[c] += (mask_binary == c).sum()

    # Avoid division by zero
    class_counts = np.maximum(class_counts, 1.0)

    # Inverse frequency: weight ∝ 1 / count
    weights = 1.0 / class_counts
    weights = weights / weights.sum() * num_classes  # Normalize to sum to num_classes

    return torch.from_numpy(weights).float()

In [ ]:
# Smoke test for loss functions with binary classes
B, C, H, W = 2, 2, 256, 256  # 2 classes for binary segmentation
logits = torch.randn(B, C, H, W, device=device)
targets = torch.randint(0, C, (B, H, W), device=device)

# Test DiceLoss
dice_loss = DiceLoss(num_classes=C)
dice = dice_loss(logits, targets)
print(f"Dice Loss: {dice.item():.4f}")

# Test CombinedLoss
combined = CombinedLoss(num_classes=C, weight_ce=0.5, weight_dice=0.5)
loss = combined(logits, targets)
print(f"Combined Loss: {loss.item():.4f}")

# Backward should work
loss.backward()
print("✓ Backward pass OK")

## Part 3: U-Net Architecture

U-Net is a fully convolutional network with an encoder-decoder structure and skip connections.

### Architecture Overview

```
Input [B, 3, 1024, 1024]
    |
    v
Encoder (4 levels)
    |- Level 1: Conv->BN->ReLU x2, then MaxPool
    |- Level 2: Conv->BN->ReLU x2, then MaxPool
    |- Level 3: Conv->BN->ReLU x2, then MaxPool
    +- Level 4: Conv->BN->ReLU x2, then MaxPool
    |
    v
Bottleneck (highest compression, full context)
    |
    v
Decoder (4 levels, with skip connections from encoder)
    |- Level 1: Upsample + concat skip + Conv->BN->ReLU x2
    |- Level 2: Upsample + concat skip + Conv->BN->ReLU x2
    |- Level 3: Upsample + concat skip + Conv->BN->ReLU x2
    +- Level 4: Upsample + concat skip + Conv->BN->ReLU x2
    |
    v
Output [B, 2, 1024, 1024] (logits: 0=background, 1=glomerulus)
```

### Why Skip Connections?

Skip connections preserve fine-grained spatial information from the encoder in the decoder, enabling pixel-level precision.

### Binary Output Head

The final Conv2d outputs 2 channels (background vs glomerulus) instead of 5 multi-class outputs. This simplification allows the model to focus on glomerulus detection independent of subtype classification, which is handled by Model 2.

In [ ]:
class DoubleConv(nn.Module):
    """Two consecutive Conv2d->BN->ReLU blocks."""

    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout) if dropout > 0 else nn.Identity(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout) if dropout > 0 else nn.Identity(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [ ]:
class Down(nn.Module):
    """Encoder step: MaxPool2d -> DoubleConv."""

    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels, dropout=dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [ ]:
class Up(nn.Module):
    """Decoder step: bilinear upsample -> concat skip -> DoubleConv."""

    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        # After concat with skip: in_channels + in_channels/2
        self.conv = DoubleConv(in_channels + in_channels // 2, out_channels, dropout=dropout)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)
        # Pad if spatial dimensions don't match (can happen with odd input sizes)
        if x.shape != skip.shape:
            diff_h = skip.shape[2] - x.shape[2]
            diff_w = skip.shape[3] - x.shape[3]
            x = torch.nn.functional.pad(x, (diff_w // 2, diff_w - diff_w // 2, diff_h // 2, diff_h - diff_h // 2))
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        return x

In [ ]:
class UNet(nn.Module):
    """
    U-Net for binary glomeruli segmentation.

    Args:
        in_channels: Number of input channels (default 3 for RGB)
        num_classes: Number of output classes (default 2: background and glomerulus)
        depth: Number of encoder/decoder levels (default 4)
        base_channels: Number of filters in first conv block (default 64)
        dropout: Dropout rate (default 0.0)
    """

    def __init__(
        self,
        in_channels: int = 3,
        num_classes: int = 2,
        depth: int = 4,
        base_channels: int = 64,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.in_channels = in_channels
        self.num_classes = num_classes
        self.depth = depth
        self.base_channels = base_channels

        # Encoder
        self.enc0 = DoubleConv(in_channels, base_channels, dropout=dropout)
        self.down = nn.ModuleList()
        for i in range(depth):
            in_ch = base_channels * (2 ** i)
            out_ch = base_channels * (2 ** (i + 1))
            self.down.append(Down(in_ch, out_ch, dropout=dropout))

        # Bottleneck
        bottleneck_ch = base_channels * (2 ** depth)
        self.bottleneck = DoubleConv(bottleneck_ch, bottleneck_ch, dropout=dropout)

        # Decoder
        self.up = nn.ModuleList()
        for i in range(depth, 0, -1):
            in_ch = base_channels * (2 ** i)
            out_ch = base_channels * (2 ** (i - 1))
            self.up.append(Up(in_ch, out_ch, dropout=dropout))

        # Final output head
        self.final_conv = nn.Conv2d(base_channels, num_classes, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: [B, 3, H, W] (Z-score encoded images)

        Returns:
            logits: [B, num_classes, H, W] (no activation)
        """
        # Encoder with skip connections
        skips = []
        out = self.enc0(x)
        skips.append(out)

        for down_block in self.down:
            out = down_block(out)
            skips.append(out)

        # Bottleneck
        out = self.bottleneck(out)

        # Decoder (reverse order, pop skips)
        for i, up_block in enumerate(self.up):
            skip = skips[-(i + 1)]
            out = up_block(out, skip)

        # Final output (logits)
        logits = self.final_conv(out)
        return logits

In [ ]:
# Test U-Net instantiation with binary classes
model = UNet(in_channels=3, num_classes=2, depth=4, base_channels=64, dropout=0.1)
model = model.to(device)
print(f"U-Net created. Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Test forward pass
x = torch.randn(1, 3, 1024, 1024, device=device)
with torch.no_grad():
    out = model(x)
print(f"Input shape: {x.shape}, Output shape: {out.shape}")
assert out.shape == (1, 2, 1024, 1024), f"Expected (1, 2, 1024, 1024), got {out.shape}"
print("✓ U-Net forward pass OK")

## Part 4: Training Pipeline

Complete training system with:
- Dynamic DataLoader worker calculation (based on available RAM)
- Binary Segmentation Metrics (Accuracy, Precision, Recall, F1, ROC-AUC)
- Augmentation pipelines (train-specific)
- Optimizer: AdamW with weight decay
- LR Scheduler: Linear warmup (5 epochs) -> Cosine annealing
- Checkpointing: saves best model (by Val F1 or Accuracy) + last model
- TensorBoard logging

### Training Workflow

1. Load data (train/val/test splits stratified by biopsia)
2. Create model, optimizer, scheduler
3. Train for N epochs:
   - Forward pass on batch
   - Compute loss (BCE + Dice with class weights for binary classification)
   - Backward pass + gradient clipping
   - Optimizer step
   - Validate after each epoch (compute binary metrics on val set)
   - Save checkpoint if best model found
4. Final evaluation on test set
5. Save metrics report

In [ ]:
def compute_dataloader_workers() -> int:
    """
    Compute optimal number of DataLoader workers based on available RAM.

    Each worker holds ~2 tiles in memory (current + prefetch).
    Returns: number of workers (0-8 range typically)
    """
    try:
        import psutil
        available_gb = psutil.virtual_memory().available / (1024 ** 3)
    except ImportError:
        available_gb = 2.0

    # Float32 RGB tile: (1024*1024*3*4) bytes = ~12MB
    tile_gb = (1024 * 1024 * 3 * 4) / (1024 ** 3)

    # Each worker holds ~2 tiles, use 30% of RAM for workers
    workers = int((available_gb * 0.3) / (tile_gb * 2))
    workers = max(0, min(workers, os.cpu_count() or 4))

    return workers

print(f"Optimal DataLoader workers: {compute_dataloader_workers()}")

In [ ]:
class MeanIoUMetric:
    """Compute Mean Intersection over Union (mIoU) metric."""

    def __init__(self, num_classes: int, ignore_background: bool = False):
        self.num_classes = num_classes
        self.ignore_background = ignore_background
        self.reset()

    def reset(self):
        self.intersection = np.zeros(self.num_classes)
        self.union = np.zeros(self.num_classes)

    def update(self, pred: torch.Tensor, target: torch.Tensor):
        """
        Args:
            pred: [B, C, H, W] logits
            target: [B, H, W] class IDs
        """
        # Convert logits to predictions
        pred_labels = torch.argmax(pred, dim=1)  # [B, H, W]

        # Move to CPU for numpy
        pred_labels = pred_labels.cpu().numpy()
        target = target.cpu().numpy()

        # Compute intersection and union per class
        for c in range(self.num_classes):
            pred_c = (pred_labels == c)
            target_c = (target == c)

            self.intersection[c] += np.logical_and(pred_c, target_c).sum()
            self.union[c] += np.logical_or(pred_c, target_c).sum()

    def compute(self) -> Tuple[float, dict]:
        """
        Returns:
            (mean_iou, per_class_iou_dict)
        """
        per_class_iou = {}
        iou_scores = []

        for c in range(self.num_classes):
            if self.union[c] == 0:
                iou = 0.0
            else:
                iou = self.intersection[c] / self.union[c]

            if not (self.ignore_background and c == 0):
                iou_scores.append(iou)

            per_class_iou[f'class_{c}'] = float(iou)

        mean_iou = float(np.mean(iou_scores)) if iou_scores else 0.0
        return mean_iou, per_class_iou

In [ ]:
class BinarySegmentationMetrics:
    """
    Compute binary segmentation metrics: Accuracy, Precision, Recall, F1, ROC-AUC.
    
    For a binary classification task where class 0 = background, class 1 = glomerulus.
    """

    def __init__(self):
        self.reset()

    def reset(self):
        """Reset all accumulators."""
        self.tp = 0  # True Positives (class 1 correctly predicted as 1)
        self.tn = 0  # True Negatives (class 0 correctly predicted as 0)
        self.fp = 0  # False Positives (class 0 incorrectly predicted as 1)
        self.fn = 0  # False Negatives (class 1 incorrectly predicted as 0)
        
        # For ROC-AUC: store predictions and targets
        self.all_probs = []  # Probability of class 1 (positive class)
        self.all_targets = []  # Ground truth labels (0 or 1)

    def update(self, pred: torch.Tensor, target: torch.Tensor):
        """
        Args:
            pred: [B, 2, H, W] logits from model
            target: [B, H, W] ground truth class IDs (0 or 1)
        """
        # Convert logits to softmax probabilities
        probs = torch.nn.functional.softmax(pred, dim=1)  # [B, 2, H, W]
        
        # Get probability of positive class (class 1)
        prob_pos = probs[:, 1, :, :]  # [B, H, W], values in [0, 1]
        
        # Get hard predictions (argmax)
        pred_labels = torch.argmax(pred, dim=1)  # [B, H, W]

        # Move to CPU for numpy
        pred_labels = pred_labels.cpu().numpy()
        target = target.cpu().numpy()
        prob_pos = prob_pos.cpu().numpy()

        # Flatten
        pred_flat = pred_labels.flatten()
        target_flat = target.flatten()
        prob_flat = prob_pos.flatten()

        # Update confusion matrix
        # TP: target=1, pred=1
        self.tp += np.logical_and(target_flat == 1, pred_flat == 1).sum()
        # TN: target=0, pred=0
        self.tn += np.logical_and(target_flat == 0, pred_flat == 0).sum()
        # FP: target=0, pred=1
        self.fp += np.logical_and(target_flat == 0, pred_flat == 1).sum()
        # FN: target=1, pred=0
        self.fn += np.logical_and(target_flat == 1, pred_flat == 0).sum()

        # Store for ROC-AUC
        self.all_probs.extend(prob_flat)
        self.all_targets.extend(target_flat)

    def compute(self) -> dict:
        """
        Compute all metrics.
        
        Returns:
            dict with keys: accuracy, precision, recall, f1, roc_auc
        """
        metrics = {}

        # Accuracy: (TP + TN) / Total
        total = self.tp + self.tn + self.fp + self.fn
        if total > 0:
            metrics['accuracy'] = float((self.tp + self.tn) / total)
        else:
            metrics['accuracy'] = 0.0

        # Precision: TP / (TP + FP)
        if (self.tp + self.fp) > 0:
            metrics['precision'] = float(self.tp / (self.tp + self.fp))
        else:
            metrics['precision'] = 0.0

        # Recall: TP / (TP + FN)
        if (self.tp + self.fn) > 0:
            metrics['recall'] = float(self.tp / (self.tp + self.fn))
        else:
            metrics['recall'] = 0.0

        # F1: 2 × (Precision × Recall) / (Precision + Recall)
        if (metrics['precision'] + metrics['recall']) > 0:
            metrics['f1'] = float(
                2 * (metrics['precision'] * metrics['recall']) / 
                (metrics['precision'] + metrics['recall'])
            )
        else:
            metrics['f1'] = 0.0

        # ROC-AUC: area under ROC curve using sklearn
        if len(self.all_targets) > 0 and len(set(self.all_targets)) > 1:
            try:
                from sklearn.metrics import roc_auc_score
                metrics['roc_auc'] = float(roc_auc_score(self.all_targets, self.all_probs))
            except Exception as e:
                print(f"Warning: Could not compute ROC-AUC: {e}")
                metrics['roc_auc'] = 0.0
        else:
            metrics['roc_auc'] = 0.0

        return metrics

In [ ]:
def get_transforms(size: int = 1024):
    """Return train/val augmentation pipelines."""
    train_transform = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.75),
        A.Transpose(p=0.5),
        A.Rotate(limit=15, p=0.3),
        A.GaussNoise(var_limit=(10, 50), p=0.2),
        A.GaussianBlur(blur_limit=3, p=0.2),
    ], additional_targets={'mask': 'mask'})

    val_transform = A.Compose([])

    return train_transform, val_transform

train_transforms, val_transforms = get_transforms()
print("✓ Augmentation pipelines created")

In [ ]:
def train_epoch(
    model: nn.Module,
    dataloader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device,
) -> float:
    """
    Train for one epoch.

    Returns:
        avg_loss: Average training loss over all batches
    """
    model.train()
    total_loss = 0.0
    num_batches = 0

    for images, masks in dataloader:
        images = images.to(device)
        masks = masks.to(device)

        # Forward pass
        logits = model(images)
        loss = criterion(logits, masks)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    avg_loss = total_loss / num_batches if num_batches > 0 else 0.0
    return avg_loss

In [ ]:
def eval_epoch(
    model: nn.Module,
    dataloader,
    criterion: nn.Module,
    device: torch.device,
) -> Tuple[float, dict]:
    """
    Evaluate for one epoch using binary segmentation metrics.

    Returns:
        (avg_loss, metrics_dict)
        
    metrics_dict contains: accuracy, precision, recall, f1, roc_auc
    """
    model.eval()
    total_loss = 0.0
    num_batches = 0
    metrics_tracker = BinarySegmentationMetrics()

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device)

            logits = model(images)
            loss = criterion(logits, masks)

            total_loss += loss.item()
            num_batches += 1

            metrics_tracker.update(logits, masks)

    avg_loss = total_loss / num_batches if num_batches > 0 else 0.0
    metrics = metrics_tracker.compute()

    return avg_loss, metrics

In [ ]:
def train_unet(
    epochs: int = 50,
    batch_size: int = 4,
    lr: float = 1e-3,
    images_dir: str = 'Salidas/Estandarizados',
    output_dir: str = 'checkpoints',
    num_workers: int = None,
    seed: int = 42,
    warmup_epochs: int = 5,
):
    """
    Complete training pipeline for U-Net binary glomeruli segmentation.
    
    Args:
        epochs: Number of training epochs
        batch_size: Batch size for training
        lr: Learning rate
        images_dir: Path to standardized images
        output_dir: Directory to save checkpoints
        num_workers: DataLoader workers (auto-calculated if None)
        seed: Random seed
        warmup_epochs: Warmup epochs for LR scheduler
    """
    # Setup
    torch.manual_seed(seed)
    np.random.seed(seed)

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # TensorBoard
    run_name = f"unet_binary_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    log_dir = output_dir / run_name
    writer = SummaryWriter(str(log_dir))

    print(f"Run: {run_name}")
    print(f"Device: {device}")
    print(f"Output dir: {output_dir}")

    # Data
    print("\nLoading data...")
    train_transforms, val_transforms = get_transforms()

    if num_workers is None:
        num_workers = compute_dataloader_workers()
        print(f"Auto-calculated DataLoader workers: {num_workers}")
    else:
        print(f"Using specified DataLoader workers: {num_workers}")

    train_loader, val_loader, test_loader = create_dataloaders(
        images_dir=images_dir,
        batch_size=batch_size,
        num_workers=num_workers,
        seed=seed,
        train_transforms=train_transforms,
        val_transforms=val_transforms,
    )

    print(f"Train: {len(train_loader.dataset)} tiles | "
          f"Val: {len(val_loader.dataset)} tiles | "
          f"Test: {len(test_loader.dataset)} tiles")

    # Model (binary: 2 classes)
    print("\nCreating model...")
    model = UNet(in_channels=3, num_classes=2, depth=4, base_channels=64, dropout=0.1)
    model = model.to(device)
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    # Class weights (inverse frequency for 2 classes: background vs glomerulus)
    try:
        class_weights = compute_class_weights(images_dir, num_classes=2)
        print(f"Class weights (background, glomerulus): {class_weights.tolist()}")
        class_weights = class_weights.to(device)
    except Exception as e:
        print(f"Warning: Could not compute class weights: {e}. Using uniform weights.")
        class_weights = None

    # Loss and optimizer
    criterion = CombinedLoss(num_classes=2, weight_ce=0.5, weight_dice=0.5,
                             class_weights=class_weights)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    # LR scheduler: linear warmup + cosine annealing
    total_steps = len(train_loader) * epochs
    warmup_steps = len(train_loader) * warmup_epochs
    scheduler = torch.optim.lr_scheduler.ChainedScheduler([
        LinearLR(optimizer, start_factor=0.1, total_iters=warmup_steps),
        CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps),
    ])

    best_val_f1 = 0.0
    train_history = []

    # Training loop
    print("\nStarting training...")
    for epoch in range(epochs):
        print(f"\n{'='*60}")
        print(f"Epoch [{epoch+1}/{epochs}]")
        print(f"{'='*60}")

        # Train
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        print(f"Train Loss: {train_loss:.4f}")

        # Val
        val_loss, val_metrics = eval_epoch(model, val_loader, criterion, device)
        print(f"Val Loss: {val_loss:.4f}")
        print(f"Val Metrics:")
        print(f"  Accuracy:  {val_metrics['accuracy']:.4f}")
        print(f"  Precision: {val_metrics['precision']:.4f}")
        print(f"  Recall:    {val_metrics['recall']:.4f}")
        print(f"  F1:        {val_metrics['f1']:.4f}")
        print(f"  ROC-AUC:   {val_metrics['roc_auc']:.4f}")

        # Step scheduler
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        print(f"LR: {current_lr:.2e}")

        # Checkpoint
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_f1': best_val_f1,
        }

        # Save last model
        torch.save(checkpoint, output_dir / f'{run_name}_last.pth')

        # Save best model (by F1 score)
        if val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']
            checkpoint['best_val_f1'] = best_val_f1
            torch.save(checkpoint, output_dir / f'{run_name}_best.pth')
            print(f"✓ Best model saved! (Val F1: {val_metrics['f1']:.4f})")

        # Log to TensorBoard
        writer.add_scalar('loss/train', train_loss, epoch)
        writer.add_scalar('loss/val', val_loss, epoch)
        writer.add_scalar('metric/val_accuracy', val_metrics['accuracy'], epoch)
        writer.add_scalar('metric/val_precision', val_metrics['precision'], epoch)
        writer.add_scalar('metric/val_recall', val_metrics['recall'], epoch)
        writer.add_scalar('metric/val_f1', val_metrics['f1'], epoch)
        writer.add_scalar('metric/val_roc_auc', val_metrics['roc_auc'], epoch)
        writer.add_scalar('lr', current_lr, epoch)

        # Track history
        train_history.append({
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_metrics': val_metrics,
        })

    writer.close()

    # Final evaluation on test set
    print(f"\n{'='*60}")
    print("Final evaluation on test set...")
    print(f"{'='*60}")

    # Load best model
    best_ckpt = torch.load(output_dir / f'{run_name}_best.pth', map_location=device)
    model.load_state_dict(best_ckpt['model_state_dict'])

    test_loss, test_metrics = eval_epoch(model, test_loader, criterion, device)
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Metrics:")
    print(f"  Accuracy:  {test_metrics['accuracy']:.4f}")
    print(f"  Precision: {test_metrics['precision']:.4f}")
    print(f"  Recall:    {test_metrics['recall']:.4f}")
    print(f"  F1:        {test_metrics['f1']:.4f}")
    print(f"  ROC-AUC:   {test_metrics['roc_auc']:.4f}")

    # Save final report
    report = {
        'run_name': run_name,
        'model_config': {
            'in_channels': 3,
            'num_classes': 2,
            'depth': 4,
            'base_channels': 64,
            'dropout': 0.1,
        },
        'training_config': {
            'epochs': epochs,
            'batch_size': batch_size,
            'learning_rate': lr,
            'warmup_epochs': warmup_epochs,
        },
        'final_metrics': {
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_metrics': val_metrics,
            'test_loss': test_loss,
            'test_metrics': test_metrics,
        },
        'training_history': train_history,
    }

    with open(output_dir / f'{run_name}_report.json', 'w') as f:
        json.dump(report, f, indent=2)

    print(f"\n✓ Training complete!")
    print(f"Checkpoints saved to: {output_dir}")
    print(f"TensorBoard logs: tensorboard --logdir {log_dir}")
    
    return model, report

## Part 6: Usage Instructions

### Running Training

Uncomment and run the cell below to start training:

In [ ]:
# Example: Train for 5 epochs on your dataset (for testing)
# Uncomment to run:

# if Path('Salidas/Estandarizados').exists():
#     model, report = train_unet(
#         epochs=5,
#         batch_size=4,
#         lr=1e-3,
#         images_dir='Salidas/Estandarizados',
#         output_dir='checkpoints',
#         num_workers=None,  # Auto-calculate
#         seed=42,
#         warmup_epochs=2,
#     )
# else:
#     print("Run the preprocessing pipeline first:")
#     print("  python tiling_unet.py")
#     print("  python normalizacion.py")
#     print("  python estandarizacion.py")

## Part 7: Troubleshooting and System Requirements

### Troubleshooting

**"No images found in Salidas/Estandarizados"**
- Run the preprocessing pipeline first (tiling -> normalization -> standardization).

**"CUDA out of memory"**
- Reduce `batch_size` (try 2 or 1)
- Increase `--ram-fraction` in preprocessing scripts
- Use CPU training (slower but uses less VRAM)

**"Expected masks directory"**
- Ensure masks were copied by normalizacion.py and estandarizacion.py to `Salidas/Estandarizados/*/masks/`
- Verify mask naming convention: `{tile_name}_mask.png`

**"No valid image-mask pairs found"**
- Check that image and mask counts match
- Verify paths: images in `*/images/` and masks in `*/masks/`

### System Requirements

- **Python**: 3.10+
- **PyTorch**: 2.0+ (with CUDA if GPU available)
- **RAM**: 16GB minimum, 32GB+ recommended
- **GPU**: Optional but recommended (4GB VRAM minimum)
- **Dependencies**: See requirements.txt (numpy, torch, torchvision, opencv, scikit-learn, albumentations, tensorboard)

### References

- **U-Net**: Ronneberger et al., "U-Net: Convolutional Networks for Biomedical Image Segmentation" (MICCAI 2015)
- **Dice Loss**: Sørensen–Dice coefficient for segmentation evaluation
- **Class Weighting**: Inverse frequency weighting for class imbalance handling